In [3]:
import itertools

def eval_expression(expr, model):
    safe_expr = expr
    for var in sorted(model.keys(), key=len, reverse=True):
        safe_expr = safe_expr.replace(var, f" {str(model[var])} ")

    safe_expr = safe_expr.replace("<=>", " == ")
    while "=>" in safe_expr:
        safe_expr = safe_expr.replace("=>", " <= ")

    safe_expr = safe_expr.replace("∧", " and ").replace("^", " and ").replace("and", " and ")
    safe_expr = safe_expr.replace("V", " or ").replace("v", " or ").replace("or", " or ")
    safe_expr = safe_expr.replace("¬", " not ").replace("~", " not ").replace("not", " not ")

    try:
        return bool(eval(safe_expr, {"__builtins__": None}, {}))
    except Exception:
        return False

if __name__ == "__main__":
    print("--- Truth-Table Generator & Propositional Entailment ---")
    print("Operators allowed: and (∧), or (V), not (¬), =>, <=>\n")

    symbols_input = input("Enter proposition symbols separated by commas (e.g., A, B, C): ")
    symbols = [s.strip() for s in symbols_input.split(",") if s.strip()]

    print("\nEnter individual clauses/sentences of the Knowledge Base (KB). Press Enter when done:")
    kb_sentences = []
    while True:
        sentence = input("  KB sentence: ").strip()
        if not sentence:
            break
        kb_sentences.append(sentence)

    alpha = input("\nEnter the query sentence (alpha): ").strip()

    headers = symbols + kb_sentences + ["KB", "alpha"]
    row_format = "{:<10}" * len(headers)

    print("\n" + row_format.format(*headers))
    print("-" * (10 * len(headers)))

    entails = True

    for combination in itertools.product([False, True], repeat=len(symbols)):
        model = dict(zip(symbols, combination))

        kb_evals = [eval_expression(sentence, model) for sentence in kb_sentences]
        kb_total = all(kb_evals)
        alpha_eval = eval_expression(alpha, model)

        if kb_total and not alpha_eval:
            entails = False

        row_values = [str(model[sym]).lower() for sym in symbols]
        row_values += [str(val).lower() for val in kb_evals]
        row_values.append(str(kb_total).lower())
        row_values.append(str(alpha_eval).lower())

        print(row_format.format(*row_values))

    print("-" * (10 * len(headers)))
    if entails:
        print(f"\nSuccess! The Knowledge Base entails alpha (KB ⊨ {alpha})")
    else:
        print(f"\nFailed! The Knowledge Base does NOT entail alpha (KB ⊭ {alpha})")


--- Truth-Table Generator & Propositional Entailment ---
Operators allowed: and (∧), or (V), not (¬), =>, <=>

Enter proposition symbols separated by commas (e.g., A, B, C): A,B,C

Enter individual clauses/sentences of the Knowledge Base (KB). Press Enter when done:
  KB sentence: A V C
  KB sentence: B V ¬C
  KB sentence: 

Enter the query sentence (alpha): A V B

A         B         C         A V C     B V ¬C    KB        alpha     
----------------------------------------------------------------------
false     false     false     false     true      false     false     
false     false     true      true      false     false     false     
false     true      false     false     true      false     true      
false     true      true      true      true      true      true      
true      false     false     true      true      true      true      
true      false     true      true      false     false     true      
true      true      false     true      true      true      true